# 02 - Reusable pipeline and honest baselines (Days 2-3)

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 50); pd.set_option('display.max_colwidth', 120)
from cortexhr import config as C, data_loading as D, audit as A, features as F, modeling as M, evaluation as E, reporting as R, runs

## Day 2: safe columns and cutoff-safe attendance/payroll features

In [2]:
S = runs.load_model_frame(include_monthly=True)
S['join_report']

{'rows_before_join': 2383, 'rows_after_join': 2383, 'cardinality': 'attrition(1) -> aggregates(1), validated one_to_one', 'attendance_matched_rows': 383, 'payroll_matched_rows': 383, 'unmatched_rows_attendance': 2000, 'unmatched_rows_payroll': 2000, 'rows_dropped': 0, 'window_months': 3, 'temporal_cutoff': 'only complete months with month_end <= observation_date; last `window` such months', 'payroll_exact_duplicates_removed': 15, 'rows_before': 2389, 'exact_duplicates_dropped': 6, 'rows_after': 2383}

In [3]:
print('Inputs:', len(S['X'].columns), 'columns; target in X?', C.TARGET in S['X'].columns)
print('Dropped for leakage/ids/dates/metadata:', C.FORBIDDEN_AS_INPUT)

Inputs: 43 columns; target in X? False
Dropped for leakage/ids/dates/metadata: ['left_within_12m', 'row_id', 'employee_id', 'observation_date', 'exit_date', 'resignation_date', 'exit_reason', 'exit_interview_done', 'fnf_settlement_amount', 'voluntary_left_within_12m', 'source']


### `build_pipeline()` and the 3-row smoke test
The pipeline cleans impossible values, adds derived features (department-relative pay etc.), imputes, scales and one-hot encodes. Everything it learns comes from the rows passed to `fit`.

In [4]:
pipe = F.build_pipeline(True).fit(S['Xtr'], S['ytr'])
raw3 = F.representative_rows(S['df'])
Z = pipe.transform(raw3[F.input_columns(True)])
print('3 raw rows ->', Z.shape, '| all finite:', bool(np.isfinite(Z.to_numpy()).all()))
raw3[['row_id','tenure_months','work_location','last_engagement_score']]

3 raw rows -> (3, 137) | all finite: True


     row_id  tenure_months work_location  last_engagement_score
108     109             -4       Chennai                    5.0
0         1            164          Pune                    2.5
1         2            162         Erode                    2.4

In [5]:
Z.iloc[:, :8]

     job_level  tenure_months  total_experience_years  previous_employers_count  commute_distance_km  comp_ratio  raises_received  last_raise_pct
108  -0.440634      -0.241674               -0.142866                 -0.184753             0.036646   -0.072364         1.058797       -0.753788
0     1.318210       4.670827                5.073470                 -0.184753            -0.027103   -0.336829         2.294223       -0.938350
1     0.438788       4.596955                2.030608                 -0.184753            -0.282097   -1.328571         2.294223       -0.177033

In [6]:
t = raw3[F.input_columns(True)].copy(); t.iloc[0, t.columns.get_loc('role_id')] = 'R999'; t.iloc[1, t.columns.get_loc('department_id')] = np.nan
print('unseen category + missing department handled:', bool(np.isfinite(pipe.transform(t).to_numpy()).all()))

unseen category + missing department handled: True


In [7]:
pd.DataFrame(F.FEATURE_DOCS, columns=['feature', 'source', 'definition', 'temporal cutoff'])

                        feature                       source                                                                                  definition                            temporal cutoff
0            pay_vs_dept_median                      derived  annual pay / median pay of same department+job level; medians fitted on training rows only               snapshot at observation_date
1        raise_stagnation_ratio                      derived                                               months_since_last_raise / (tenure_months + 1)               snapshot at observation_date
2                never_promoted                      derived              1 if promotions_received == 0 (explains the blank months_since_last_promotion)               snapshot at observation_date
3                log_annual_pay                      derived                       log1p(annual_fixed_pay) after blanking pay-typos (>3x role mid-point)               snapshot at observation_date
4   att_attendance_r

**Why fit on training data only (3 sentences).** 1) Medians, means, categories and department pay medians describe the data; if they are computed on test rows, test information leaks into training and scores look better than reality. 2) A new employee scored tomorrow is, by definition, not in the data the pipeline learned from, so the honest test is data the pipeline never saw. 3) `Pipeline.fit()` inside every cross-validation fold guarantees that.

## Day 3: stratified split, shared folds, baselines

In [8]:
print('train', S['Xtr'].shape, 'test', S['Xte'].shape, '| leaver rate train/test:', round(S['ytr'].mean(), 4), round(S['yte'].mean(), 4))
print('fold sizes (train, valid):', [(len(a), len(b)) for a, b in S['folds']])

train (1906, 43) test (477, 43) | leaver rate train/test: 0.1464 0.1468
fold sizes (train, valid): [(1524, 382), (1525, 381), (1525, 381), (1525, 381), (1525, 381)]


In [9]:
o = runs.run_baselines(write=False)
o['comparison'].round(3)

                                               model  roc_auc  roc_auc_sd  benchmark_roc_auc  delta_roc_auc  pr_auc  benchmark_pr_auc  precision  benchmark_precision  recall  benchmark_recall  \
0                               Dummy (always stays)    0.500       0.000              0.500          0.000   0.146             0.146      0.000                0.000   0.000             0.000   
1                     Logistic Regression (balanced)    0.698       0.041              0.713         -0.015   0.312             0.332      0.248                0.260   0.573             0.630   
2                  Decision Tree (depth 4, balanced)    0.630       0.033              0.650         -0.020   0.237             0.256      0.216                0.234   0.538             0.524   
3                      Random Forest (300, balanced)    0.703       0.013              0.715         -0.012   0.308             0.336      0.469                0.523   0.050             0.089   
4  LEAKY Logistic Regress

In [10]:
o['fold_metrics'][['model', 'fold', 'roc_auc', 'pr_auc', 'precision', 'recall', 'f1', 'accuracy', 'precision_defined']].round(3).head(20)

                                model  fold  roc_auc  pr_auc  precision  recall     f1  accuracy  precision_defined
0                Dummy (always stays)     1    0.500   0.147      0.000   0.000  0.000     0.853              False
1                Dummy (always stays)     2    0.500   0.144      0.000   0.000  0.000     0.856              False
2                Dummy (always stays)     3    0.500   0.147      0.000   0.000  0.000     0.853              False
3                Dummy (always stays)     4    0.500   0.147      0.000   0.000  0.000     0.853              False
4                Dummy (always stays)     5    0.500   0.147      0.000   0.000  0.000     0.853              False
5      Logistic Regression (balanced)     1    0.696   0.298      0.254   0.554  0.348     0.696               True
6      Logistic Regression (balanced)     2    0.673   0.249      0.233   0.545  0.326     0.675               True
7      Logistic Regression (balanced)     3    0.740   0.397      0.267 

Undefined precision (no positive predictions, as for the Dummy) is reported as 0 via `zero_division=0` and flagged by `precision_defined`.

In [11]:
print(o['tree']); o['imbalance'].round(3)

{'train_auc_in_sample': 1.0, 'cv_auc_out_of_fold': 0.550207850057167}


                           model  roc_auc_mean  roc_auc_sd  pr_auc_mean  pr_auc_sd  precision_mean  recall_mean  f1_mean  accuracy_mean  top_precision_mean  top_recall_mean  \
0      LR balanced class weights         0.698       0.041        0.312      0.056           0.248        0.573    0.346          0.681               0.369            0.258   
1                  LR no weights         0.700       0.041        0.316      0.043           0.440        0.118    0.183          0.846               0.385            0.269   
2  LR + SMOTE (train folds only)         0.682       0.041        0.301      0.062           0.238        0.538    0.329          0.678               0.364            0.254   

   folds_with_undefined_precision  
0                               0  
1                               0  
2                               0  

In [12]:
o['comparison_core'][['model','roc_auc','benchmark_roc_auc','delta_roc_auc']].round(3)

                               model  roc_auc  benchmark_roc_auc  delta_roc_auc
0               Dummy (always stays)    0.500              0.500          0.000
1     Logistic Regression (balanced)    0.705              0.713         -0.008
2  Decision Tree (depth 4, balanced)    0.624              0.650         -0.026
3      Random Forest (300, balanced)    0.698              0.715         -0.017

### Five-sentence interpretation

In [13]:
print(*[f'{i}. {s}' for i, s in enumerate(o['interpretation'], 1)], sep='\n')

1. Logistic regression reaches a cross-validated ROC-AUC of 0.698 (+/-0.041), 0.015 below the benchmark 0.713; a gap this small is believable because my CV uses 80% of the rows and different fold boundaries, and the fold-to-fold spread is itself about 0.04.
2. The always-stay Dummy scores 0.854 accuracy but ROC-AUC 0.500 and recall 0, so accuracy alone would have called it a success while it finds no leavers.
3. The balanced logistic regression catches 57% of leavers at precision 25% (benchmark 63% / 26%), meaning most flagged people are false alarms: that is the price of finding more leavers on a 14.6% base rate.
4. The depth-4 tree (0.630) is weaker than the logistic regression and the forest (0.703) is about equal in ranking but almost never says 'will leave' at a 0.5 threshold (recall 5%); an unlimited tree scores 1.00 AUC on its own training rows but only 0.55 out-of-fold because it memorises noise.
5. No honest model exceeds 0.90 (the highest is 0.703), so the leakage alarm is qu

In [14]:
o['summary'][['model', 'roc_auc_mean', 'leakage_check']]

                                               model  roc_auc_mean                                          leakage_check
0                               Dummy (always stays)      0.500000                                                     ok
1                     Logistic Regression (balanced)      0.697767                                                     ok
2                  Decision Tree (depth 4, balanced)      0.629683                                                     ok
3                      Random Forest (300, balanced)      0.703477                                                     ok
4  LEAKY Logistic Regression (adds exit information)      1.000000  INVESTIGATE: ROC-AUC > 0.90 (leakage/evaluation bug?)

Warnings captured during the run (none are hidden):

In [15]:
o['warnings'] or 'none'

'none'